# Exam Analysis — Overview and risk effect

This notebook loads the exam data and analyses:
1. General results summary
2. Correlation between taking risks (answering more questions) and the final grade
3. Answer breakdown per student

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Estilo
plt.rcParams.update({
    'figure.facecolor': '#1a1a2e', 'axes.facecolor': '#16213e',
    'axes.edgecolor': '#e94560', 'axes.labelcolor': '#e0e0e0',
    'text.color': '#e0e0e0', 'xtick.color': '#e0e0e0',
    'ytick.color': '#e0e0e0', 'grid.color': '#2a2a4a', 'grid.alpha': 0.5,
    'font.family': 'sans-serif', 'font.size': 11,
})

COLORS = {
    'correct': '#00e676', 'wrong': '#e94560', 'unanswered': '#888888',
    'scatter': '#53d8fb', 'trend': '#ffc947', 'pass_line': '#00e676',
}

STUDENT_COLORS = [
    '#e94560', '#53d8fb', '#ffc947', '#00e676', '#a29bfe', '#ff6b6b',
    '#55efc4', '#fd79a8', '#74b9ff', '#fab1a0', '#dfe6e9', '#636e72',
]

# Configuración del examen
POINTS_CORRECT = 0.11
POINTS_WRONG = -0.05
PASS_THRESHOLD = 5.0

print('✅ Configuración cargada')

## 1. Data loading

In [ ]:
CSV_PATH = Path('.') / '134_36018173_ZE3IFC005200_MP5072_A-Examen 1a evaluación-cualificacións.csv'

def parse_answer(val) -> str:
    if isinstance(val, (int, float)):
        return 'correct' if val > 0 else ('wrong' if val < 0 else 'unanswered')
    s = str(val).strip().strip("'").strip('\u2018')
    if s in ('-', '', 'nan'):
        return 'unanswered'
    try:
        num = float(s.replace(',', '.'))
        return 'correct' if num > 0 else ('wrong' if num < 0 else 'unanswered')
    except ValueError:
        return 'unanswered'

def parse_score(val) -> float:
    if isinstance(val, (int, float)):
        return float(val)
    s = str(val).strip().strip("'").strip('\u2018')
    if s in ('-', '', 'nan'):
        return 0.0
    try:
        return float(s.replace(',', '.'))
    except ValueError:
        return 0.0

# Cargar CSV
df = pd.read_csv(CSV_PATH)

# Eliminar fila 'Media xeral'
df = df[df['Apelidos'].str.strip() != 'Media xeral'].dropna(subset=['Apelidos']).copy()

# Columnas de preguntas
q_cols = [c for c in df.columns if c.startswith('P.')]
NUM_QUESTIONS = len(q_cols)

# Nombre del alumno
df['Alumno'] = df['Nome'].astype(str).str.strip() + ' ' + df['Apelidos'].astype(str).str.strip()
df['Nombre_Corto'] = df['Nome'].astype(str).str.strip()

# Desambiguar nombres repetidos
for name in df['Nombre_Corto'].value_counts().index:
    mask = df['Nombre_Corto'] == name
    if mask.sum() > 1:
        df.loc[mask, 'Nombre_Corto'] = (
            df.loc[mask, 'Nome'].str.strip() + ' ' +
            df.loc[mask, 'Apelidos'].str.split().str[0]
        )

# Nota original
grade_col = [c for c in df.columns if 'ualificaci' in c][0]
df['Nota_Original'] = df[grade_col].apply(parse_score)

# Matrices de respuestas y puntuaciones
answer_types = pd.DataFrame(index=df.index)
scores = pd.DataFrame(index=df.index)
for col in q_cols:
    answer_types[col] = df[col].apply(parse_answer)
    scores[col] = df[col].apply(parse_score)

n_students = len(df)
print(f'📄 Alumnos: {n_students}')
print(f'📝 Preguntas: {NUM_QUESTIONS}')
print(f'📊 Nota media: {df["Nota_Original"].mean():.2f}')
print(f'✅ Aprobados: {(df["Nota_Original"] >= PASS_THRESHOLD).sum()}/{n_students}')
print(f'\nPuntuación: acierto={POINTS_CORRECT}, error={POINTS_WRONG}')

## 2. Grade summary

In [ ]:
# Tabla de notas ordenada
resumen = df[['Alumno', 'Nota_Original']].sort_values('Nota_Original', ascending=False).copy()
resumen['Resultado'] = resumen['Nota_Original'].apply(
    lambda x: '✅ Aprobado' if x >= PASS_THRESHOLD else '❌ Suspenso'
)
resumen.columns = ['Alumno', 'Nota', 'Resultado']
resumen

In [ ]:
# Histograma de notas
fig, ax = plt.subplots(figsize=(10, 5))
bins = np.arange(0, 11, 1)
ax.hist(df['Nota_Original'], bins=bins, color='#0f3460', edgecolor='white',
        linewidth=1.2, alpha=0.85, rwidth=0.85)
ax.axvline(x=PASS_THRESHOLD, color=COLORS['pass_line'], linestyle='--',
           linewidth=2.5, alpha=0.8, label='Aprobado (≥5)')
mean_val = df['Nota_Original'].mean()
ax.axvline(x=mean_val, color='white', linestyle=':', linewidth=1.5, alpha=0.5)
ax.text(mean_val + 0.1, ax.get_ylim()[1] * 0.85, f'μ={mean_val:.1f}', fontsize=10, color='white')
n_pass = (df['Nota_Original'] >= PASS_THRESHOLD).sum()
ax.set_title(f'Distribución de Notas — SAA 1ª Evaluación ({n_pass}/{n_students} aprobados)',
             fontweight='bold', fontsize=14)
ax.set_xlabel('Nota')
ax.set_ylabel('Nº Alumnos')
ax.set_xlim(0, 10)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Risk-taking effect analysis

Is there a correlation between answering more questions (taking more penalty risk) and the final grade?

In [ ]:
# Estadísticas por alumno
stats = pd.DataFrame()
stats['Alumno'] = df['Alumno'].values
stats['Nombre'] = df['Nombre_Corto'].values
stats['Nota'] = df['Nota_Original'].values
stats['Aciertos'] = (answer_types[q_cols] == 'correct').sum(axis=1).values
stats['Errores'] = (answer_types[q_cols] == 'wrong').sum(axis=1).values
stats['Sin_Resp'] = (answer_types[q_cols] == 'unanswered').sum(axis=1).values
stats['Respondidas'] = stats['Aciertos'] + stats['Errores']
stats['Tasa_Acierto'] = (stats['Aciertos'] / stats['Respondidas'] * 100).round(1)
stats['Aprobado'] = stats['Nota'] >= PASS_THRESHOLD

stats.sort_values('Nota', ascending=False)

In [ ]:
# Correlaciones
corr_errors = np.corrcoef(stats['Errores'], stats['Nota'])[0, 1]
corr_answered = np.corrcoef(stats['Respondidas'], stats['Nota'])[0, 1]
corr_accuracy = np.corrcoef(stats['Tasa_Acierto'], stats['Nota'])[0, 1]

print(f'Correlación (errores ↔ nota):             {corr_errors:+.3f}')
print(f'Correlación (preguntas respondidas ↔ nota): {corr_answered:+.3f}')
print(f'Correlación (tasa de acierto ↔ nota):       {corr_accuracy:+.3f}')
print()
if corr_errors < -0.3:
    print('⚠️  Más errores se asocia con peores notas → arriesgar penaliza')
elif corr_errors > 0.3:
    print('💡 Los que más arriesgan también aciertan más')
else:
    print('ℹ️  No hay correlación clara entre errores y nota')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
fig.suptitle('Análisis del Riesgo: Relación entre Respuestas y Calificación',
             fontweight='bold', fontsize=15)

# Plot 1: Errores vs Nota
ax = axes[0]
ax.scatter(stats['Errores'], stats['Nota'], c=COLORS['scatter'], s=120,
           edgecolors='white', linewidth=1.5, zorder=5, alpha=0.9)
z = np.polyfit(stats['Errores'], stats['Nota'], 1)
p = np.poly1d(z)
x_line = np.linspace(stats['Errores'].min() - 1, stats['Errores'].max() + 1, 100)
ax.plot(x_line, p(x_line), '--', color=COLORS['trend'], linewidth=2, alpha=0.7,
        label=f'Tendencia (r={corr_errors:+.2f})')
ax.axhline(y=PASS_THRESHOLD, color=COLORS['pass_line'], linestyle=':', linewidth=1.5,
           alpha=0.7, label='Aprobado (5.0)')
for _, r in stats.iterrows():
    ax.annotate(r['Nombre'].split()[0], (r['Errores'], r['Nota']),
                textcoords='offset points', xytext=(5, 8), fontsize=7, alpha=0.8)
ax.set_xlabel('Nº de Respuestas Erróneas')
ax.set_ylabel('Nota Final (/10)')
ax.set_title('Errores vs Nota')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Plot 2: Respondidas vs Nota
ax = axes[1]
ax.scatter(stats['Respondidas'], stats['Nota'], c='#e94560', s=120,
           edgecolors='white', linewidth=1.5, zorder=5, alpha=0.9)
z2 = np.polyfit(stats['Respondidas'], stats['Nota'], 1)
p2 = np.poly1d(z2)
x2 = np.linspace(stats['Respondidas'].min() - 1, stats['Respondidas'].max() + 1, 100)
ax.plot(x2, p2(x2), '--', color=COLORS['trend'], linewidth=2, alpha=0.7,
        label=f'Tendencia (r={corr_answered:+.2f})')
ax.axhline(y=PASS_THRESHOLD, color=COLORS['pass_line'], linestyle=':', linewidth=1.5,
           alpha=0.7, label='Aprobado (5.0)')
for _, r in stats.iterrows():
    ax.annotate(r['Nombre'].split()[0], (r['Respondidas'], r['Nota']),
                textcoords='offset points', xytext=(5, 8), fontsize=7, alpha=0.8)
ax.set_xlabel('Nº de Preguntas Respondidas')
ax.set_ylabel('Nota Final (/10)')
ax.set_title('Respondidas vs Nota')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Plot 3: Desglose por alumno
ax = axes[2]
idx = np.arange(n_students)
sort_order = stats.sort_values('Nota').index
s = stats.loc[sort_order]
names_short = [n.split()[0] for n in s['Nombre']]
w = 0.6
ax.barh(idx, s['Aciertos'].values, w, color=COLORS['correct'], label='Aciertos', alpha=0.85)
ax.barh(idx, s['Errores'].values, w, left=s['Aciertos'].values,
        color=COLORS['wrong'], label='Errores', alpha=0.85)
ax.barh(idx, s['Sin_Resp'].values, w,
        left=s['Aciertos'].values + s['Errores'].values,
        color=COLORS['unanswered'], label='Sin responder', alpha=0.6)
ax.set_yticks(idx)
ax.set_yticklabels(names_short, fontsize=8)
ax.set_xlabel('Nº de Preguntas')
ax.set_title('Desglose de Respuestas')
ax.legend(fontsize=7, loc='lower right')
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## Conclusions of the risk analysis

The data from this exam allow us to determine:

1. **Correlation between errors and grade**: Indicates whether the current penalty (−0.05 per error, ratio ~1:2.2 relative to the +0.11 for a correct answer) is disproportionately penalising students who attempt more questions.
2. **Correlation answered–grade**: Indicates whether answering more questions tends to improve or worsen the grade.
3. **Accuracy as predictor**: The precision (% of correct answers out of attempted) is usually the best predictor of the final grade, more so than the total number of answers.
4. **The visual breakdown** allows identifying students who took high risks (many attempted + many errors) vs conservative students (many unanswered).